# Current Model Experiments

This notebook is for real model experiments only. Production implementations stay under `code/models`, feature code stays under `code/features`, and postprocess code stays under `code/portfolio`.

Experiment rule: use rolling k-fold validation, use all available history before each fold, and use at least 100 boosting rounds.

## Current Method K-Fold

- Feature set: `158+39+window_multi_cross+xsec`
- Input window: 12 complete trading weeks
- Validation mode: `rolling_kfold` via `code/utils/validation.py`
- Folds: 4 rolling validation blocks before holdout-test
- Training data: all available history before each fold train end
- Boosting rounds: 100
- Models: `xgb_rank_pairwise`, `lambdarankic`
- Device: CPU for reproducible experiment execution


In [ ]:
import json
import os
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import xgboost as xgb

from code.models.xgboost.config import config
from code.models.xgboost.loss import lambdarankic_objective, topk_return_metrics, xgb_rank_ic_metric, xgb_rank_return_metrics
from code.models.xgboost.train import make_dmatrix, choose_top5_iteration, target_range
from code.features.baseline import preprocess_stock_data_samples
from code.features.windows import feature_num_for_window
from code.utils.runtime_split import build_stock_data_samples, load_market_data
from code.utils.validation import build_validation_plan

random.seed(42)
np.random.seed(42)
os.environ['PYTHONHASHSEED'] = '42'

MODEL_NAMES = ['xgb_rank_pairwise', 'lambdarankic']
NUM_ROUND = 100
OUTPUT_PATH = Path('output/current_methods_kfold_experiment.json')


In [ ]:
base_model = config['model_params']['xgb_rank_pairwise']
input_window = int(base_model['input_window'])
feature_type = base_model['feature_type']
feature_num = feature_num_for_window(input_window, feature_type)

raw_df = load_market_data(config)
stock_ids = sorted(raw_df['股票代码'].unique())
stockid2idx = {sid: idx for idx, sid in enumerate(stock_ids)}
plan = build_validation_plan(raw_df, config)
folds = plan.rolling_splits()

# 构造 holdout-test 之前的全部历史样本，保证每个 rolling fold 都能从同一特征表切片。
samples = build_stock_data_samples(raw_df, plan.start_date, plan.holdout_test_target_start, input_window)
all_df, features = preprocess_stock_data_samples(samples, feature_num, stockid2idx)
all_df = all_df.dropna(subset=['label']).sort_values(['日期', '股票代码']).reset_index(drop=True)

experiment_data = {
    'feature_num': feature_num,
    'features': len(features),
    'input_window': input_window,
    'rounds': NUM_ROUND,
    'folds': len(folds),
    'all_sample_range': target_range(all_df),
    'all_rows': len(all_df),
    'models': MODEL_NAMES,
}
experiment_data


In [ ]:
def train_eval_fold_model(fold, name, train_df, val_df):
    dtrain, train_groups = make_dmatrix(train_df, features)
    dval, val_groups = make_dmatrix(val_df, features)
    params = {
        key: value
        for key, value in config['model_params'][name].items()
        if key not in {'input_window', 'feature_type'}
    }
    params['device'] = 'cpu'
    params['tree_method'] = 'hist'
    params['nthread'] = 8
    params['verbosity'] = 0
    params['disable_default_eval_metric'] = 1
    obj = lambdarankic_objective if name == 'lambdarankic' else None
    evals_result = {}
    started = time.time()
    booster = xgb.train(
        params=params,
        dtrain=dtrain,
        num_boost_round=NUM_ROUND,
        evals=[(dtrain, 'train'), (dval, 'validation')],
        obj=obj,
        custom_metric=xgb_rank_return_metrics,
        maximize=True,
        evals_result=evals_result,
        verbose_eval=False,
    )
    selection = choose_top5_iteration([float(v) for v in evals_result['validation']['top5_return']])
    val_pred = booster.predict(dval, iteration_range=(0, int(selection['best_iteration']) + 1))
    val_top5 = topk_return_metrics(val_pred, dval.get_label(), val_groups, top_k=5)
    val_top10 = topk_return_metrics(val_pred, dval.get_label(), val_groups, top_k=10)
    return {
        'fold': fold.name,
        'model': name,
        'rounds': NUM_ROUND,
        'best_iteration': int(selection['best_iteration']),
        'best_validation_top5_return': float(selection['top5_return']),
        'validation_rank_ic': float(xgb_rank_ic_metric(val_pred, dval)[1]),
        'validation_top5_return': float(val_top5['pred_top5_return_avg']),
        'validation_top10_return': float(val_top10['pred_top10_return_avg']),
        'validation_top5_by_week': val_top5['pred_top5_group_returns'],
        'validation_top10_by_week': val_top10['pred_top10_group_returns'],
        'train_groups': len(train_groups),
        'validation_groups': len(val_groups),
        'train_rows': int(dtrain.num_row()),
        'validation_rows': int(dval.num_row()),
        'seconds': time.time() - started,
    }


In [ ]:
rows = []
started = time.time()
for fold in folds:
    train_df = all_df[(all_df['日期'] >= fold.train_start) & (all_df['日期'] < fold.train_end)].copy()
    val_df = all_df[(all_df['日期'] >= fold.validation_start) & (all_df['日期'] < fold.validation_end)].copy()
    for model_name in MODEL_NAMES:
        rows.append(train_eval_fold_model(fold, model_name, train_df, val_df))

summary_df = pd.DataFrame(rows)
metric_cols = ['best_validation_top5_return', 'validation_rank_ic', 'validation_top5_return', 'validation_top10_return']
summary = summary_df.groupby('model')[metric_cols].agg(['mean', 'std', 'min', 'max'])
summary.columns = ['_'.join(col).rstrip('_') for col in summary.columns]
summary = summary.reset_index()
payload = {
    'experiment_data': experiment_data,
    'folds': [fold.__dict__ for fold in folds],
    'rows': rows,
    'summary': summary.to_dict('records'),
    'elapsed_seconds': time.time() - started,
}
OUTPUT_PATH.parent.mkdir(exist_ok=True)
OUTPUT_PATH.write_text(json.dumps(payload, ensure_ascii=False, indent=2, default=str), encoding='utf-8')
summary


## Captured Result

Executed in this workspace. Full artifact written to `output/current_methods_kfold_experiment.json`.

Data shape: 568 features, 39825 rows, target sample range `2023-04-17` to `2026-05-18`.

| model | mean val top5 | std val top5 | min val top5 | max val top5 | mean RankIC | mean val top10 |
| --- | ---: | ---: | ---: | ---: | ---: | ---: |
| xgb_rank_pairwise | 0.033533 | 0.020243 | 0.013608 | 0.058668 | 0.082699 | 0.016713 |
| lambdarankic | 0.019609 | 0.016711 | 0.007585 | 0.043216 | 0.023394 | 0.010975 |

Fold-level validation Top5:

| fold | xgb_rank_pairwise | lambdarankic |
| --- | ---: | ---: |
| rolling_1 | 0.058668 | 0.043216 |
| rolling_2 | 0.021258 | 0.007585 |
| rolling_3 | 0.013608 | 0.007951 |
| rolling_4 | 0.040599 | 0.019683 |

Takeaway: with rolling k-fold, full history, and 100 rounds, `xgb_rank_pairwise` is stronger than `lambdarankic` on every fold for validation Top5 and has the higher average RankIC.

In [ ]:
captured = json.loads(OUTPUT_PATH.read_text(encoding='utf-8'))
pd.DataFrame(captured['summary'])
